# Underwater enhancement: classical pipeline (no weights)

Tests a **deterministic, non-learned** enhancement pipeline as an alternative to FUnIE-GAN. The stages are applied in this order:

- **Grey-world white balance**: corrects the colour cast
- **Edge-redness reduction**: damps the red cast observed towards the frame edges
- **UDCP dehazing** (Underwater Dark Channel Prior): removes haze and backscatter
- **CLAHE**: recovers local contrast

There are no pretrained weights, no downloads and no domain-mismatch risk, so this always runs. It is also a useful baseline specifically for **Moreton Bay**: the water is turbid and sediment-heavy (Brisbane River outflow), with a brownish-green cast quite different from the blue-green tropical and Red Sea water most learned enhancement models were trained on. If a classical, untrained method does about as well as (or better than) a learned one on this footage, that is a meaningful data point about domain mismatch.

The pipeline code is in `utils/classical_utils.py`. The controlled comparison against expert annotations is in `zeroshot/coral_enhancement_comparison.ipynb`.

In [ ]:
import sys
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

sys.path.append("../utils")   # shared modules live in the repository's utils folder
from classical_utils import classical_enhance, grey_world_white_balance, udcp_dehaze, apply_clahe, reduce_edge_redness

VIDEO_PATH = "../data/bluerov_videos/Shortened.mp4"
FRAME_SAMPLE_EVERY_N_SEC = 5
OUT_DIR = "outputs/classical"

## Sanity check: each pipeline stage on one frame

In [ ]:
assert os.path.exists(VIDEO_PATH), f"Video not found at {VIDEO_PATH}"

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise IOError(f"Could not open video: {VIDEO_PATH}")

ret, test_frame_bgr = cap.read()
if not ret:
    raise IOError("Could not read a test frame.")
cap.release()

test_frame = Image.fromarray(cv2.cvtColor(test_frame_bgr, cv2.COLOR_BGR2RGB))
test_arr = np.array(test_frame)

wb = grey_world_white_balance(test_arr)
edge_corrected = reduce_edge_redness(wb, strength=0.5, start_radius=0.55)
dehazed = udcp_dehaze(edge_corrected, patch_size=15, omega=0.6, t0=0.2)
final = apply_clahe(dehazed, clip_limit=1.5)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
axes[0].imshow(test_arr); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(wb); axes[1].set_title("+ White balance"); axes[1].axis("off")
axes[2].imshow(edge_corrected); axes[2].set_title("+ Edge Redness Reduce"); axes[2].axis("off")
axes[3].imshow(dehazed); axes[3].set_title("+ UDCP dehaze"); axes[3].axis("off")
axes[4].imshow(final); axes[4].set_title("+ CLAHE (final)"); axes[4].axis("off")
plt.tight_layout()
plt.show()

## Process sampled frames from the video

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_interval = int(fps * FRAME_SAMPLE_EVERY_N_SEC)

pairs = []  # (timestamp, original_pil, enhanced_pil)
frame_idx = 0

while True:
    ret, frame_bgr = cap.read()
    if not ret:
        break

    if frame_idx % frame_interval == 0:
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(frame_rgb)

        enhanced = classical_enhance(image, use_clahe=False)
        pairs.append((frame_idx / fps, image, enhanced))
        print(f"Enhanced frame at t={frame_idx/fps:.1f}s")

    frame_idx += 1

cap.release()
print(f"\nDone. Collected {len(pairs)} before/after pairs.")

## Visualise before and after

In [ ]:
def show_before_after(timestamp, original, enhanced, method_name="Classical pipeline"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(original); axes[0].set_title(f"Original (t={timestamp:.1f}s)"); axes[0].axis("off")
    axes[1].imshow(enhanced); axes[1].set_title(f"{method_name} enhanced"); axes[1].axis("off")
    plt.tight_layout()
    plt.show()

for i, (t, orig, enh) in enumerate(pairs):
    if i % 3 == 0:
        show_before_after(t, orig, enh)

## Save enhanced frames to disk

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
for i, (t, orig, enh) in enumerate(pairs):
    enh.save(f"{OUT_DIR}/frame_{i:04d}_t{t:.1f}s.png")
print(f"Saved {len(pairs)} enhanced frames to {OUT_DIR}/")

## Run the classically enhanced frames through Coralscapes

A qualitative check of how the Coralscapes V1 segmentation responds to classically enhanced frames from the full Myora Bay video. There is no ground truth here, so nothing is scored; the quantitative evaluation is in `zeroshot/`.

In [ ]:
import torch
from coral_utils import load_coralscapes_model, segment_image, colourise_mask, coverage_stats

SEGMENT_VIDEO_PATH = "../data/bluerov_videos/Myora-BlueROV.mp4"
SEGMENT_EVERY_N_SEC = 50

# Apple GPU (MPS) when available, otherwise CPU
device = "mps" if torch.backends.mps.is_available() else "cpu"
preprocessor, model = load_coralscapes_model(device)


def show_segmentation(image, pred, alpha=0.6):
    colour_mask = colourise_mask(pred)
    image_np = np.array(image)
    overlay = (image_np * (1 - alpha) + colour_mask * alpha).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(image_np); axes[0].set_title("Enhanced frame"); axes[0].axis("off")
    axes[1].imshow(colour_mask); axes[1].set_title("Segmentation"); axes[1].axis("off")
    axes[2].imshow(overlay); axes[2].set_title("Overlay"); axes[2].axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Extract and enhance frames
cap = cv2.VideoCapture(SEGMENT_VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_interval = int(fps * SEGMENT_EVERY_N_SEC)

seg_pairs = []  # (timestamp, original_pil, enhanced_pil)
frame_idx = 0

while True:
    ret, frame_bgr = cap.read()
    if not ret:
        break

    if frame_idx % frame_interval == 0:
        image = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
        enhanced = classical_enhance(image, use_clahe=False)
        seg_pairs.append((frame_idx / fps, image, enhanced))
        print(f"Enhanced frame at t={frame_idx/fps:.1f}s")

    frame_idx += 1

cap.release()
print(f"\nDone. Collected {len(seg_pairs)} before/after pairs.")

# Segment the enhanced frames
first_frame_checked = False

for t, original, enhanced in seg_pairs:
    pred = segment_image(enhanced, preprocessor, model, device)
    show_segmentation(enhanced, pred)

    # One-time sanity check on the predicted class id range
    if not first_frame_checked:
        print(f"Predicted class id range: {pred.min()}-{pred.max()}")
        first_frame_checked = True

    print(f"\n--- t={t:.1f}s (enhanced) ---")
    for name, pct in coverage_stats(pred)[:5]:
        print(f"  {name:28s} {pct:5.2f}%")